# JapyScript Compiler Analysis

This notebook documents profiling, benchmark, and correctness analyses
for the JapyScript compiler pipeline. Results are regenerated on every
release and committed alongside the source.

In [ ]:
import sys, time, statistics
from pathlib import Path

BENCH_DIR = Path('..') / 'tests' / 'bench'
print('benchmark directory:', BENCH_DIR.resolve())
print('python:', sys.version.split()[0])

In [ ]:
def time_stage(fn, *args, runs=20):
    samples = []
    for _ in range(runs):
        t0 = time.perf_counter()
        fn(*args)
        samples.append(time.perf_counter() - t0)
    return {
        'mean_ms': statistics.mean(samples) * 1000,
        'p95_ms': sorted(samples)[int(runs * 0.95)] * 1000,
    }

## Lexer throughput

The lexer is a single-pass state machine. We measure tokens per second on
the conformance suite to detect regressions introduced by new keywords.

In [ ]:
from compiler.lexer import tokenize

SAMPLE = '\n'.join([
    'def f(x):',
    '    return x * 2',
    '',
    'result = f(21)',
]) * 200

def bench_lexer():
    return tokenize(SAMPLE, '<bench>')

print(time_stage(bench_lexer))

## Notes

Batch 1 establishes baseline numbers. Batches 2–4 extend the benchmark
with parser, checker, and VM stages.